# 05a - Evaluate General Strength

CPU only. Compare completed broad, specialist, optional self-play and classical candidates. These are research matches, with no submission export checks.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Discover Candidates

Load or create the immutable candidate registry, independently of 05b. Complete configured training or explicitly skip stages before registration. Optional no-RL selections and additional trusted checkpoints can be named in research_workflow. Self-play is included only when its configured league has completed.

In [ ]:
import torch
torch.set_num_threads(1)
from chess_rl.candidate_registry import load_candidate_registry
registry = load_candidate_registry(PROJECT_ROOT, cfg)
candidates = registry["candidates"]
for candidate in candidates:
    print(candidate["id"], candidate["kind"], candidate["checkpoint"])

## Play Common Development Opponents

All candidates use the same clocks, opening pairs and greedy/minimax/original-classical opponents. The development results support selection; held-out games wait until 06a freezes one winner.

In [ ]:
from chess_rl.research_workflow import general_evaluation
general = general_evaluation(PROJECT_ROOT, cfg, candidates)
for name, opponents in general["results"].items():
    print(name, {opponent: result["score"] for opponent, result in opponents.items()})
print("Saved:", PROJECT_ROOT / "results" / cfg["run_id"] / "general_evaluation.json")